# Hybrid vs Alex QUBO Size Probe

Generate one Steiner Tree Problem instance, convert it with Daghan's hybrid formulation and Alex's ordering formulation, then compare variable and quadratic-interaction counts. No plots: edit the parameters and re-run the cells manually.

In [12]:
from pathlib import Path
import sys

import dimod
import openjij as oj
from tqdm import tqdm

def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "SteinerTreeProblemQUBO").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root containing SteinerTreeProblemQUBO")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from SteinerTreeProblemQUBO.random_problem_generator import (
    generate_random_steiner_tree,
    generate_geometric_steiner_tree,
    generate_erdos_renyi_steiner_tree,
    generate_grid_steiner_tree,
)
from SteinerTreeProblemQUBO.sparsity_problem_generator import generate_sparsity_steiner_tree
from SteinerTreeProblemQUBO.MyFormulization.steiner_to_oj_qubo_hybrid import steiner_to_oj_qubo_hybrid
from SteinerTreeProblemQUBO.AlexFowler.steiner_to_oj_qubo_alex import steiner_to_oj_qubo_alex

REPO_ROOT

PosixPath('/Users/daghanerdonmez/Desktop/evvifing/boun/cmpe/491-492/finale')

In [13]:
GENERATORS = {
    "sparsity": generate_sparsity_steiner_tree,
    "random_connected": generate_random_steiner_tree,
    "erdos_renyi": generate_erdos_renyi_steiner_tree,
    "geometric": generate_geometric_steiner_tree,
    "grid": generate_grid_steiner_tree,
}

FORMULATIONS = {
    "hybrid": steiner_to_oj_qubo_hybrid,
    "alex": steiner_to_oj_qubo_alex,
}

def make_problem(generator_name="sparsity", **kwargs):
    return GENERATORS[generator_name](**kwargs)

def default_constraint_weight(problem):
    # Large enough for size probing; the exact value should not change variable counts.
    return max(weight for _, _, weight in problem.edges) + 1

def qubo_size_stats(Q):
    variables = set()
    linear_terms = 0
    quadratic_pairs = set()
    nonzero_terms = 0

    for (u, v), bias in Q.items():
        if bias == 0:
            continue
        variables.add(u)
        variables.add(v)
        nonzero_terms += 1
        if u == v:
            linear_terms += 1
        else:
            quadratic_pairs.add(tuple(sorted((u, v))))

    return {
        "variables": len(variables),
        "linear_terms": linear_terms,
        "quadratic_interactions": len(quadratic_pairs),
        "nonzero_qubo_terms": nonzero_terms,
    }

def format_table(rows, columns):
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    rule = "-+-".join("-" * widths[column] for column in columns)
    body = [
        " | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns)
        for row in rows
    ]
    return "\n".join([header, rule, *body])

def compare_problem(problem, constraint_weight=None, print_qubo_sizes=True):
    if constraint_weight is None:
        constraint_weight = default_constraint_weight(problem)

    rows = []
    qubos = {}
    for model_name, converter in FORMULATIONS.items():
        Q, offset = converter(problem, constraint_weight)
        qubos[model_name] = Q
        rows.append({
            "model": model_name,
            **qubo_size_stats(Q),
            "offset": offset,
        })

    by_model = {row["model"]: row for row in rows}
    hybrid = by_model["hybrid"]
    alex = by_model["alex"]
    delta = {
        "variables_delta_hybrid_minus_alex": hybrid["variables"] - alex["variables"],
        "quadratic_delta_hybrid_minus_alex": hybrid["quadratic_interactions"] - alex["quadratic_interactions"],
        "hybrid_wins_variables": hybrid["variables"] < alex["variables"],
        "hybrid_wins_quadratics": hybrid["quadratic_interactions"] < alex["quadratic_interactions"],
        "hybrid_wins_both": (
            hybrid["variables"] < alex["variables"]
            and hybrid["quadratic_interactions"] < alex["quadratic_interactions"]
        ),
    }

    if print_qubo_sizes:
        print(f"nodes={len(problem.nodes)}, edges={len(problem.edges)}, terminals={len(problem.terminals)}")
        print(f"terminal set: {problem.terminals}")
        print(f"constraint_weight={constraint_weight}")
        print()
        print(format_table(rows, ["model", "variables", "quadratic_interactions", "linear_terms", "nonzero_qubo_terms"]))
        print()
        print(delta)

    return {
        "problem": problem,
        "constraint_weight": constraint_weight,
        "rows": rows,
        "delta": delta,
        "qubos": qubos,
    }

In [ ]:
def solve_with_openjij(problem, model_name, constraint_weight=None, num_reads=1000, show_stats=True, show_progress=False, **sampler_kwargs):
    if model_name not in FORMULATIONS:
        raise ValueError(f"Unknown model_name: {model_name}")
    if num_reads < 1:
        raise ValueError("num_reads must be at least 1")

    if constraint_weight is None:
        constraint_weight = default_constraint_weight(problem)

    converter = FORMULATIONS[model_name]
    qubo, offset = converter(problem, constraint_weight)
    bqm = dimod.BinaryQuadraticModel.from_qubo(qubo, offset=offset)

    if show_stats:
        print(f"[{model_name}] Problem converted to QUBO")
        print(f"[{model_name}] Number of variables: {bqm.num_variables}")
        print(f"[{model_name}] Number of interactions: {bqm.num_interactions}")

    sampler = oj.SQASampler()
    responses = []
    sampling_runs = range(num_reads)
    if show_progress:
        sampling_runs = tqdm(
            sampling_runs,
            total=num_reads,
            desc=f"Sampling {model_name}",
            unit="read",
            file=sys.stdout,
        )

    for _ in sampling_runs:
        responses.append(sampler.sample_qubo(qubo, num_reads=1, **sampler_kwargs))

    response = dimod.concatenate(responses)
    best = response.first

    return {
        "model": model_name,
        "constraint_weight": constraint_weight,
        "qubo": qubo,
        "offset": offset,
        "bqm": bqm,
        "response": response,
        "best_sample": best.sample,
        "best_energy_without_offset": best.energy,
        "best_energy_with_offset": best.energy + offset,
    }

def solve_both_with_openjij(problem, constraint_weight=None, num_reads=1000, show_stats=True, **sampler_kwargs):
    results = {}
    for model_name in FORMULATIONS:
        results[model_name] = solve_with_openjij(
            problem,
            model_name=model_name,
            constraint_weight=constraint_weight,
            num_reads=num_reads,
            show_stats=show_stats,
            show_progress=True,
            **sampler_kwargs,
        )
    return results

def summarize_openjij_results(results):
    rows = []
    for model_name, result in results.items():
        rows.append({
            "model": model_name,
            "variables": result["bqm"].num_variables,
            "quadratic_interactions": result["bqm"].num_interactions,
            "best_energy_with_offset": result["best_energy_with_offset"],
            "active_bits_in_best_sample": sum(result["best_sample"].values()),
        })
    print(format_table(rows, ["model", "variables", "quadratic_interactions", "best_energy_with_offset", "active_bits_in_best_sample"]))


In [19]:
def solve_with_dimod_sa(problem, model_name, constraint_weight=None, num_reads=1000, show_stats=True, show_progress=False, **sampler_kwargs):
    if model_name not in FORMULATIONS:
        raise ValueError(f"Unknown model_name: {model_name}")
    if num_reads < 1:
        raise ValueError("num_reads must be at least 1")

    if constraint_weight is None:
        constraint_weight = default_constraint_weight(problem)

    converter = FORMULATIONS[model_name]
    qubo, offset = converter(problem, constraint_weight)
    bqm = dimod.BinaryQuadraticModel.from_qubo(qubo, offset=offset)

    if show_stats:
        print(f"[{model_name}] Problem converted to BQM for dimod SA")
        print(f"[{model_name}] Number of variables: {bqm.num_variables}")
        print(f"[{model_name}] Number of interactions: {bqm.num_interactions}")

    sampler = dimod.SimulatedAnnealingSampler()
    responses = []
    sampling_runs = range(num_reads)
    if show_progress:
        sampling_runs = tqdm(
            sampling_runs,
            total=num_reads,
            desc=f"Dimod SA {model_name}",
            unit="read",
            file=sys.stdout,
        )

    for _ in sampling_runs:
        responses.append(sampler.sample(bqm, num_reads=1, **sampler_kwargs))

    response = dimod.concatenate(responses)
    best = response.first

    return {
        "model": model_name,
        "constraint_weight": constraint_weight,
        "qubo": qubo,
        "offset": offset,
        "bqm": bqm,
        "response": response,
        "best_sample": best.sample,
        "best_energy_without_offset": best.energy - offset,
        "best_energy_with_offset": best.energy,
    }

def solve_both_with_dimod_sa(problem, constraint_weight=None, num_reads=1000, show_stats=True, **sampler_kwargs):
    results = {}
    for model_name in FORMULATIONS:
        results[model_name] = solve_with_dimod_sa(
            problem,
            model_name=model_name,
            constraint_weight=constraint_weight,
            num_reads=num_reads,
            show_stats=show_stats,
            show_progress=True,
            **sampler_kwargs,
        )
    return results

def summarize_dimod_sa_results(results):
    rows = []
    for model_name, result in results.items():
        rows.append({
            "model": model_name,
            "variables": result["bqm"].num_variables,
            "quadratic_interactions": result["bqm"].num_interactions,
            "best_energy_with_offset": result["best_energy_with_offset"],
            "active_bits_in_best_sample": sum(result["best_sample"].values()),
        })
    print(format_table(rows, ["model", "variables", "quadratic_interactions", "best_energy_with_offset", "active_bits_in_best_sample"]))


## Try One Instance

Edit `GENERATOR_NAME` and `PARAMS`, then run the cell. The default uses your sparsity generator because it lines up with the benchmark scripts.

In [20]:
# GENERATOR_NAME = "sparsity"
# PARAMS = {
#     "node_count": 10,
#     "terminal_count": 4,
#     "extra_edge_probability": 0.3,
#     "weight_range": (1, 100),
#     "seed": 0,
# }

# Other examples:
# GENERATOR_NAME = "random_connected"
# PARAMS = {"node_count": 10, "terminal_count": 4, "extra_edge_probability": 0.3, "weight_range": (1, 100), "seed": 0}
#
# GENERATOR_NAME = "erdos_renyi"
# PARAMS = {"node_count": 10, "terminal_count": 4, "edge_probability": 0.3, "weight_range": (1, 100), "seed": 0}
#
GENERATOR_NAME = "geometric"
PARAMS = {"node_count": 50, "terminal_count": 5, "connectivity": "knn", "k": 5, "max_weight": 100, "seed": 0}

# GENERATOR_NAME = "grid"
# PARAMS = {"rows": 3, "cols": 4, "terminal_count": 4, "weight_range": (1, 100), "seed": 0}

problem = make_problem(GENERATOR_NAME, **PARAMS)
comparison = compare_problem(problem)

nodes=50, edges=154, terminals=5
terminal set: ['v40', 'v21', 'v12', 'v15', 'v1']
constraint_weight=101

model  | variables | quadratic_interactions | linear_terms | nonzero_qubo_terms
-------+-----------+------------------------+--------------+-------------------
hybrid | 2753      | 44975                  | 2753         | 47728             
alex   | 1477      | 57972                  | 1429         | 59401             

{'variables_delta_hybrid_minus_alex': 1276, 'quadratic_delta_hybrid_minus_alex': -12997, 'hybrid_wins_variables': False, 'hybrid_wins_quadratics': True, 'hybrid_wins_both': False}


In [ ]:
# Inspect the generated STP instance if you want to sanity-check it.
print("nodes:", problem.nodes)
print("terminals:", problem.terminals)
print("edges:")
for edge in problem.edges:
    print("  ", edge)

## Optional OpenJij Solve

This mirrors the existing `oj_solver_hybrid.py` and `oj_solver_alex.py` flow: build the QUBO, send it to `openjij.SQASampler`, then compare the best returned energies.

In [16]:
RUN_OPENJIJ = True
NUM_READS = 100
SAMPLER_KWARGS = {
    "num_sweeps": 4000,
    "trotter": 16,
}

if RUN_OPENJIJ:
    oj_results = solve_both_with_openjij(
        problem,
        constraint_weight=comparison["constraint_weight"],
        num_reads=NUM_READS,
        show_stats=True,
        **SAMPLER_KWARGS,
    )
    summarize_openjij_results(oj_results)
else:
    print("Set RUN_OPENJIJ = True to sample both formulations with OpenJij.")

[hybrid] Problem converted to QUBO
[hybrid] Number of variables: 2753
[hybrid] Number of interactions: 44975


KeyboardInterrupt: 

In [ ]:
# Inspect one best sample if you want.
# Example: model_name = "hybrid" or "alex"
model_name = "hybrid"

if "oj_results" in globals():
    print("best energy:", oj_results[model_name]["best_energy_with_offset"])
    print("active variables in best sample:")
    for var, value in oj_results[model_name]["best_sample"].items():
        if value == 1:
            print(var, value)
else:
    print("Run the OpenJij solve cell first.")

## Optional dimod Simulated Annealing

This follows the pattern in `dimod_solver.py`: convert to a BQM, do one read at a time with `dimod.SimulatedAnnealingSampler`, then compare the best returned energies.

In [22]:
RUN_DIMOD_SA = True
DIMOD_NUM_READS = 100
DIMOD_SAMPLER_KWARGS = {}

if RUN_DIMOD_SA:
    dimod_sa_results = solve_both_with_dimod_sa(
        problem,
        constraint_weight=comparison["constraint_weight"],
        num_reads=DIMOD_NUM_READS,
        show_stats=True,
        **DIMOD_SAMPLER_KWARGS,
    )
    summarize_dimod_sa_results(dimod_sa_results)
else:
    print("Set RUN_DIMOD_SA = True to sample both formulations with dimod simulated annealing.")

[hybrid] Problem converted to BQM for dimod SA
[hybrid] Number of variables: 2753
[hybrid] Number of interactions: 44975
Dimod SA hybrid: 100%|██████████| 100/100 [1:16:43<00:00, 46.04s/read]
[alex] Problem converted to BQM for dimod SA
[alex] Number of variables: 1477
[alex] Number of interactions: 57972
Dimod SA alex: 100%|██████████| 100/100 [1:35:59<00:00, 57.59s/read]
model  | variables | quadratic_interactions | best_energy_with_offset | active_bits_in_best_sample
-------+-----------+------------------------+-------------------------+---------------------------
hybrid | 2753      | 44975                  | 1415102.0               | 1103                      
alex   | 1477      | 57972                  | 348.0                   | 523                       


In [ ]:
# Inspect one best dimod SA sample if you want.
model_name = "hybrid"

if "dimod_sa_results" in globals():
    print("best energy:", dimod_sa_results[model_name]["best_energy_with_offset"])
    print("active variables in best sample:")
    for var, value in dimod_sa_results[model_name]["best_sample"].items():
        if value == 1:
            print(var, value)
else:
    print("Run the dimod simulated annealing cell first.")

## Optional Manual Sweep

This is still intentionally small and text-only. Add/remove specs in `TRIALS`, run it, and look for the first rows where `hybrid_wins_both` becomes `True`.

In [ ]:
TRIALS = [
    {"generator": "sparsity", "node_count": 6, "terminal_count": 2, "extra_edge_probability": 0.1, "seed": 0},
    {"generator": "sparsity", "node_count": 8, "terminal_count": 3, "extra_edge_probability": 0.1, "seed": 0},
    {"generator": "sparsity", "node_count": 10, "terminal_count": 4, "extra_edge_probability": 0.3, "seed": 0},
    {"generator": "sparsity", "node_count": 12, "terminal_count": 5, "extra_edge_probability": 0.3, "seed": 0},
]

summary_rows = []
for spec in TRIALS:
    spec = dict(spec)
    generator = spec.pop("generator")
    spec.setdefault("weight_range", (1, 100))

    p = make_problem(generator, **spec)
    result = compare_problem(p, print_qubo_sizes=False)
    stats = {row["model"]: row for row in result["rows"]}

    summary_rows.append({
        "generator": generator,
        "n": len(p.nodes),
        "k": len(p.terminals),
        "m": len(p.edges),
        "p_extra": spec.get("extra_edge_probability", spec.get("edge_probability", "")),
        "seed": spec.get("seed", ""),
        "hybrid_vars": stats["hybrid"]["variables"],
        "alex_vars": stats["alex"]["variables"],
        "hybrid_quad": stats["hybrid"]["quadratic_interactions"],
        "alex_quad": stats["alex"]["quadratic_interactions"],
        "hybrid_wins_both": result["delta"]["hybrid_wins_both"],
    })

print(format_table(summary_rows, [
    "generator", "n", "k", "m", "p_extra", "seed",
    "hybrid_vars", "alex_vars", "hybrid_quad", "alex_quad", "hybrid_wins_both",
]))